# XGBoost Experiments
This notebook is a lightweight sandbox for primitive test runs using XGBoost.
Run cells in order from top to bottom.

In [61]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import importlib
import preprocessing_NLP
importlib.reload(preprocessing_NLP)
from preprocessing_NLP import build_raw_dataset_nlp
from preprocessing import load_all_data

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

BASE_DIR = "datasets_full.csv/datasets_full.csv"
USERS_FILE = "users.csv"
TWEETS_FILE = "tweets.csv"

DATASETS = {
    "genuine_accounts.csv": 0,
    "fake_followers.csv": 1,
    "social_spambots_1.csv": 1,
    "social_spambots_2.csv": 1,
    "social_spambots_3.csv": 1,
    "traditional_spambots_1.csv": 1,
    "traditional_spambots_2.csv": 1,
    "traditional_spambots_3.csv": 1,
    "traditional_spambots_4.csv": 1,
}

TEXT_COLUMN = "text"

USER_FEATURES = [
    "statuses_count", "followers_count", "friends_count",
    "favourites_count", "listed_count", "default_profile",
    "default_profile_image", "verified"
]

TWEET_FEATURES = [
   "text", "reply_count", "favorite_count", "num_urls", "num_mentions"
]

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "logloss",
    "use_label_encoder": False,
    "random_state": 42,
}

print("Dataset root:", BASE_DIR)
print("Files configured:", len(DATASETS))
print("XGBoost version:", xgb.__version__)

Dataset root: datasets_full.csv/datasets_full.csv
Files configured: 9
XGBoost version: 3.2.0


In [62]:
# Block 1: Data loading (full configured datasets)
raw = build_raw_dataset_nlp(
    DATASETS,
    BASE_DIR,
    TWEET_FEATURES,
    TWEETS_FILE
)

Loaded genuine_accounts.csv: 3,474 rows
Loaded fake_followers.csv: 3,351 rows
Loaded social_spambots_1.csv: 991 rows
Loaded social_spambots_2.csv: 3,457 rows
Loaded social_spambots_3.csv: 464 rows
Loaded traditional_spambots_1.csv: 1,000 rows
Loaded traditional_spambots_2.csv: 100 rows
Loaded traditional_spambots_3.csv: 403 rows
Loaded traditional_spambots_4.csv: 1,128 rows

Total rows loaded: 14,368


In [63]:
# Block 2: Slicing and preprocessing
available_numeric = [c for c in USER_FEATURES if c in raw.columns]
if not available_numeric:
    raise ValueError("No numeric feature columns found in raw data.")

X = raw[available_numeric].copy()
X = X.apply(pd.to_numeric, errors="coerce")

numeric_tweet_features = [c for c in TWEET_FEATURES if c != "text" and c in raw.columns]
X = pd.concat([X, raw[numeric_tweet_features]], axis=1)
X = X.fillna(0)

if "text" in raw.columns:
    X["text"] = raw["text"].fillna("").astype(str)

y = raw["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print("Numeric features:", [c for c in X.columns if c != "text"])
print("Shape:", X.shape)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class balance:\n", y.value_counts(dropna=False))
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

Numeric features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'default_profile', 'default_profile_image', 'verified', 'reply_count', 'favorite_count', 'num_urls', 'num_mentions']
Shape: (14368, 13)
Train shape: (11494, 13)
Test shape: (2874, 13)
Class balance:
 label
1    10894
0     3474
Name: count, dtype: int64
scale_pos_weight: 0.3189


In [64]:
# Block 3: Training
from sklearn.preprocessing import MaxAbsScaler

tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
text_train = tfidf.fit_transform(X_train["text"])
text_test  = tfidf.transform(X_test["text"])

num_cols = [c for c in X_train.columns if c != "text"]
X_train_num = X_train[num_cols].values.astype(np.float32)
X_test_num  = X_test[num_cols].values.astype(np.float32)

# Scale numeric features to same range as TF-IDF (0-1)
scaler = MaxAbsScaler()
X_train_num = scaler.fit_transform(X_train_num)
X_test_num  = scaler.transform(X_test_num)

X_train_final = sp.hstack([X_train_num, text_train])
X_test_final  = sp.hstack([X_test_num,  text_test])

tfidf_feature_names = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
all_feature_names = num_cols + tfidf_feature_names

print(f"Final train matrix: {X_train_final.shape}")
print(f"Final test matrix:  {X_test_final.shape}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.3, 
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight,
)

model.fit(
    X_train_final, y_train,
    eval_set=[(X_test_final, y_test)],
    verbose=50,
)

booster = model.get_booster()
booster.feature_names = all_feature_names

print("\nTraining complete.")
if hasattr(model, 'best_iteration') and model.best_iteration is not None:
    print("Best iteration:", model.best_iteration)
else:
    print("No early stopping used. Total rounds:", model.n_estimators)

Final train matrix: (11494, 96)
Final test matrix:  (2874, 96)
[0]	validation_0-logloss:0.60317
[50]	validation_0-logloss:0.04817
[100]	validation_0-logloss:0.02078
[150]	validation_0-logloss:0.01610
[200]	validation_0-logloss:0.01479
[250]	validation_0-logloss:0.01405
[299]	validation_0-logloss:0.01379

Training complete.
No early stopping used. Total rounds: 300


In [65]:
# Block 4: Evaluation
pred = model.predict(X_test_final, validate_features=False)
pred_proba = model.predict_proba(X_test_final, validate_features=False)[:, 1]

print(f"accuracy:          {accuracy_score(y_test, pred):.4f}")
print(f"balanced_accuracy: {balanced_accuracy_score(y_test, pred):.4f}")
print(f"f1_macro:          {f1_score(y_test, pred, average='macro'):.4f}")
print(f"roc_auc:           {roc_auc_score(y_test, pred_proba):.4f}")
print(f"AUPRC:             {average_precision_score(y_test, pred_proba):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))

print("AUPRC:", round(average_precision_score(y_test, pred_proba), 4))

importance_df = (
    pd.DataFrame(
        model.get_booster().get_score(importance_type="gain"),
        index=["gain"]
    )
    .T.rename_axis("feature").reset_index()
    .sort_values("gain", ascending=False)
)

print("\nTop 20 features by gain:")
display(importance_df.head(20))

accuracy:          0.9962
balanced_accuracy: 0.9950
f1_macro:          0.9948
roc_auc:           0.9999
AUPRC:             1.0000

Confusion Matrix:
[[ 690    5]
 [   6 2173]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9914    0.9928    0.9921       695
           1     0.9977    0.9972    0.9975      2179

    accuracy                         0.9962      2874
   macro avg     0.9945    0.9950    0.9948      2874
weighted avg     0.9962    0.9962    0.9962      2874

AUPRC: 1.0

Top 20 features by gain:


,feature,gain
10,num_mentions,74.046997
9,num_urls,60.700863
8,favorite_count,32.267399
3,favourites_count,30.981457
7,reply_count,17.299719
1,followers_count,5.869959
0,statuses_count,4.986845
2,friends_count,2.660167
5,default_profile,1.827390
4,listed_count,1.740143


In [66]:
# Block 5: Feature importance list
importance = model.get_booster().get_score(importance_type="gain")
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)

print("Top 20 Features — XGBoost (gain):")
for i, (feature, score) in enumerate(importance_sorted[:20], 1):
    print(f"  {i:2}. {feature:<35} {score:.4f}")

Top 20 Features — XGBoost (gain):
   1. num_mentions                        74.0470
   2. num_urls                            60.7009
   3. favorite_count                      32.2674
   4. favourites_count                    30.9815
   5. reply_count                         17.2997
   6. followers_count                     5.8700
   7. statuses_count                      4.9868
   8. friends_count                       2.6602
   9. default_profile                     1.8274
  10. listed_count                        1.7401
  11. default_profile_image               0.9270


In [67]:
# Block 6: Save model
MODEL_PATH = "xgboost_bot_detector.json"
model.save_model(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

Model saved to xgboost_bot_detector.json


## Run Order
1. Cell 1: imports and constants
2. Cell 2: helper functions
3. Cell 3: full data loading
4. Cell 4: slicing and preprocessing
5. Cell 5: training
6. Cell 6: evaluation
7. Cell 7: feature importance list
8. Cell 8: save model